In [ ]:
import os, glob, subprocess
if not os.path.exists('/kaggle/working/Basma'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/us3ph/Basma.git', '/kaggle/working/Basma'], check=True)
os.chdir('/kaggle/working/Basma'); print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
import os, glob, json, subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
# data-kernel outputs (mounted at /kaggle/input/notebooks/<owner>/<slug>/): v2's four builds + the two v3 builds
mans = sorted(m for m in glob.glob('/kaggle/input/**/manifest_*.jsonl', recursive=True) if '/basma-data-' in m)
dirs = sorted({os.path.dirname(m) for m in mans}); print('DATA DIRS', dirs)
assert len(dirs) == 6, f'expected cf/recent/reals/eval + v3ai/v3real, found {mans}'
# baseline scores from v2 (same bench rows and per-image attack seeds): only rows new to v3 are scored
cache = sorted(glob.glob('/kaggle/input/**/runs/kaggle_v2/baselines/thermostatic.jsonl', recursive=True))
os.environ['BASELINE_CACHE'] = os.path.dirname(cache[0]) if cache else ''; print('BASELINE_CACHE', os.environ['BASELINE_CACHE'] or 'none')
cfg = json.load(open('configs/kaggle_2xT4_v3.json')); cfg['data_dirs'] = dirs
json.dump(cfg, open('configs/kaggle_run.json', 'w'), indent=1); print(json.dumps(cfg, indent=1))

In [ ]:
!bash scripts/kaggle_train.sh configs/kaggle_run.json

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('docs/RESULTS.md').read()))
if os.path.exists('runs/kaggle_v3/ntire/NTIRE.md'): display(Markdown(open('runs/kaggle_v3/ntire/NTIRE.md').read()))